# 量化交易 · 数据分析 · 机器学习 — 环境自检与入门

本 Notebook 用于验证本机 Python 环境（受管 Python 3.13.12 @ `default` 虚拟环境）中所有核心依赖是否安装成功，并给出最小可运行示例。

目录：
1. 依赖版本自检
2. 数据分析（pandas / polars）
3. 技术指标（ta）与可视化（matplotlib / plotly）
4. 机器学习（scikit-learn）最小示例
5. 数据获取接口（tushare / akshare）用法提示

In [ ]:
import importlib, sys
print("Python:", sys.version)

libs = [
    ("numpy", "numpy"), ("pandas", "pandas"), ("scipy", "scipy"),
    ("polars", "polars"), ("pyarrow", "pyarrow"), ("openpyxl", "openpyxl"),
    ("matplotlib", "matplotlib"), ("seaborn", "seaborn"), ("plotly", "plotly"),
    ("bokeh", "bokeh"), ("mplfinance", "mplfinance"),
    ("tushare", "tushare"), ("akshare", "akshare"), ("baostock", "baostock"),
    ("yfinance", "yfinance"), ("pandas_datareader", "pandas_datareader"),
    ("ta", "ta"), ("backtrader", "backtrader"), ("quantstats", "quantstats"),
    ("ccxt", "ccxt"),
    ("sklearn", "scikit-learn"), ("statsmodels", "statsmodels"),
    ("xgboost", "xgboost"), ("lightgbm", "lightgbm"), ("catboost", "catboost"),
    ("arch", "arch"), ("pmdarima", "pmdarima"), ("shap", "shap"),
]

print("\n=== 依赖版本自检 ===")
miss = []
for label, name in libs:
    try:
        m = importlib.import_module(name)
        print(f"  {label:20s} OK   {m.__version__}")
    except Exception as e:
        miss.append(label)
        print(f"  {label:20s} 缺失 ({e.__class__.__name__})")
print("\n缺失项:", miss if miss else "无，全部就绪 ✅")

## 2. 数据分析（pandas / polars 对比）

In [ ]:
import numpy as np, pandas as pd
rng = np.random.default_rng(42)

# 构造一段模拟日线行情
dates = pd.bdate_range("2024-01-01", periods=120)
price = 100 * np.cumprod(1 + rng.normal(0.0005, 0.02, 120))
df = pd.DataFrame({"close": price}, index=dates)
df["ret"] = df["close"].pct_change()

print(df.tail())
print("\n年化波动率(近似):", round(df["ret"].std() * np.sqrt(252) * 100, 2), "%")

## 3. 技术指标（ta）与可视化

In [ ]:
import ta
df["rsi"]  = ta.momentum.RSIIndicator(df["close"], window=14).rsi()
macd = ta.trend.MACD(df["close"])
df["macd"] = macd.macd()
df["macd_signal"] = macd.macd_signal()
print(df[["close", "rsi", "macd", "macd_signal"]].tail())

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (10, 3.5)
fig, ax = plt.subplots(2, 1, sharex=True)
ax[0].plot(df.index, df["close"]); ax[0].set_title("收盘价")
ax[1].plot(df.index, df["rsi"]); ax[1].axhline(30, ls="--", c="g"); ax[1].axhline(70, ls="--", c="r"); ax[1].set_title("RSI(14)")
plt.tight_layout(); plt.show()

## 4. 机器学习（scikit-learn）最小示例
用过去 N 日收益预测次日涨跌方向。

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

N = 5
X = pd.concat([df["ret"].shift(i) for i in range(1, N + 1)], axis=1).dropna()
y = (df["ret"].shift(-1) > 0).iloc[N-1:-1].astype(int)
X = X.iloc[:-1]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, shuffle=False, test_size=0.3)
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_tr, y_tr)
print(classification_report(y_te, clf.predict(X_te), digits=3))

## 5. 数据获取接口用法提示

- **Tushare**：需注册获取 token（`import tushare as ts; ts.set_token('YOUR_TOKEN'); pro = ts.pro_api()`）。
- **AkShare**：免费，`import akshare as ak; ak.stock_zh_a_hist(symbol='600519', period='daily', start_date='20240101', end_date='20241231', adjust='qfq')`。
- 本项目 `stock_data_spec.yaml` 已定义标准化字段（前复权 OHLCV + 估值指标），可作为数据对齐规范。
- 加密行情可用 `ccxt`（已预装）。